# L'esponente di coda mu e il test dell'Eq. 8 di Altmann

## Perche'

Altmann et al. costruiscono la loro spiegazione su una ipotesi precisa (Eq. 8): la
distribuzione degli intervalli fra occorrenze ha coda a legge di potenza

        p(tau) ~ tau^(-mu)      con      C_tau(k) = delta(k)

e da questa deriva la relazione

        gamma = 4 - mu          valida per      2 < mu < 3

Il paper usa mu = 2.4 per Guerra e pace, da cui gamma = 1.6, e misura gamma = 1.68.
La banda 2 < mu < 3 e' il regime in cui la media degli intervalli esiste ma la varianza
diverge: e' esattamente la ragione per cui sigma_tau/<tau> non converge e deriva con la
lunghezza del testo (1.26 a N = 40.000 caratteri, 3.89 sul libro intero).

Questo notebook fa due cose che nel materiale disponibile non erano mai state fatte:

1. **stima mu direttamente** con lo stimatore di Hill, invece di assumerlo;
2. **verifica l'Eq. 8** confrontando il gamma osservato con 4 - mu, su quattro corpora,
   inclusi Wikipedia e Grokipedia dove la relazione non e' mai stata controllata.

## Il dettaglio che e' facile sbagliare

Hill stima l'esponente della **funzione di sopravvivenza**, non della densita':

        se   P(tau > t) ~ t^(-alpha)      allora   p(tau) ~ tau^(-(alpha+1))

quindi

        mu = alpha_Hill + 1

Con mu = 2.4 il paper corrisponde ad alpha = 1.4.

## Contro cosa va confrontato: gamma_A2, non gamma

L'Eq. 8 assume C_tau(k) = delta(k), cioe' intervalli **scorrelati fra loro**. Nel testo
vero gli intervalli sono correlati, e il gamma osservato riceve un contributo aggiuntivo
da quelle correlazioni: il paper stesso scrive gamma >= gamma_A2.

Il null model A2 permuta gli intervalli, quindi **preserva p(tau) e distrugge C_tau(k)**:
realizza esattamente le ipotesi dell'Eq. 8. Il test corretto e' dunque

        gamma_A2   contro   4 - mu

Confrontare gamma (originale) con 4 - mu sovrastimerebbe sistematicamente, e sarebbe un
errore di lettura del paper.

## 1. Configurazione

In [ ]:
from pathlib import Path

QUI  = Path.cwd()
BASE = QUI if (QUI / "corpora_cache").is_dir() else QUI.parent
CACHE_WIKI = QUI / "cache_wikipedia"
CACHE_GROK = QUI / "cache_grokipedia"
CORPUS_V01 = QUI / "risultati_confronto" / "corpus_v01"
RIEPILOGO  = QUI / "risultati_confronto" / "dati" / "riepilogo.csv"
SEQUENZE   = QUI / "risultati_tre_corpora" / "dati" / "sequenze.csv"
CACHE_LIB  = BASE / "corpora_cache"
OUT_DIR    = QUI / "risultati_coda"

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
START_PHRASE  = "Well, Prince, so Genoa and Lucca"
PROMPT_CHARS  = 8262

N_EFF      = 60_000        # come nel notebook a tre corpora
ESCLUSI    = ["Buddhism"]
N_SEG_LETT = 20
MIN_EVENTS = 15

# --- stima della coda ---
# Con ~45 intervalli per sequenza una stima di Hill per singola sequenza avrebbe
# errore standard alpha/sqrt(k) ~ 0.4: inutilizzabile. Si mettono quindi in comune
# gli intervalli di tutte le sequenze dello stesso (corpus, tipo), normalizzando
# ciascuna sequenza per la propria media: una legge di potenza e' priva di scala,
# quindi la normalizzazione preserva l'esponente e rende i campioni commensurabili.
FRAZ_CODA   = 0.10         # frazione di code usata da Hill (k = FRAZ_CODA * n)
N_BOOT      = 300          # ripetizioni bootstrap per l'intervallo di confidenza
MIN_TAU_SEQ = 20           # intervalli minimi perche' una sequenza entri nel pool

N_LETTERS, N_FUNCTION, N_KEYWORDS, N_MATCHED = 19, 6, 7, 7
PROPER_CAP_RATIO = 0.6
RANDOM_SEED = 20260814
DPI = 150

In [ ]:
import re, math, ssl, hashlib, urllib.request
import html as htmlmod
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from scipy import stats as sps, optimize, integrate
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False
try:
    import certifi
    HAS_CERTIFI = True
except Exception:
    HAS_CERTIFI = False

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": DPI, "savefig.bbox": "tight",
                     "font.size": 10, "axes.grid": True, "grid.alpha": .25,
                     "axes.axisbelow": True})
for d in (OUT_DIR, OUT_DIR / "figure", OUT_DIR / "dati"):
    d.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(RANDOM_SEED)
print("scipy:", HAS_SCIPY, "| output ->", OUT_DIR.resolve())

## 2. Lo stimatore di Hill

Dati i valori ordinati in modo decrescente x(1) >= x(2) >= ... >= x(n), con k statistiche
d'ordine superiori:

        alpha_Hill(k) = [ (1/k) * somma_{i=1..k} ln( x(i) / x(k+1) ) ]^(-1)

e mu = alpha + 1. L'errore standard asintotico e' alpha/sqrt(k), che qui viene comunque
ricalcolato per bootstrap perche' i campioni sono messi in comune fra sequenze.

In [ ]:
def hill(x, k=None, fraz=FRAZ_CODA):
    """alpha della funzione di sopravvivenza. Ritorna (alpha, k usato)."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    n = len(x)
    if n < 30:
        return np.nan, 0
    if k is None:
        k = max(10, int(round(fraz * n)))
    k = min(k, n - 1)
    xs = np.sort(x)[::-1]
    soglia = xs[k]
    if soglia <= 0:
        return np.nan, 0
    s = np.mean(np.log(xs[:k] / soglia))
    return (1.0 / s if s > 0 else np.nan), k

def hill_mu(x, k=None, fraz=FRAZ_CODA):
    a, k_us = hill(x, k, fraz)
    return (a + 1.0 if np.isfinite(a) else np.nan), k_us

def hill_boot(x, r, nboot=N_BOOT, fraz=FRAZ_CODA):
    """mu con intervallo di confidenza al 95% per bootstrap non parametrico"""
    x = np.asarray(x, dtype=float)
    mu0, k_us = hill_mu(x, fraz=fraz)
    if not np.isfinite(mu0):
        return dict(mu=np.nan, lo=np.nan, hi=np.nan, k=0, n=len(x))
    n = len(x)
    vals = []
    for _ in range(nboot):
        m, _ = hill_mu(x[r.integers(0, n, n)], fraz=fraz)
        if np.isfinite(m):
            vals.append(m)
    if not vals:
        return dict(mu=mu0, lo=np.nan, hi=np.nan, k=k_us, n=n)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return dict(mu=float(mu0), lo=float(lo), hi=float(hi), k=int(k_us), n=int(n))

print("stimatore di Hill pronto")

## 3. Validazione su code note

Prima di applicarlo, va verificato che restituisca l'esponente giusto su campioni generati
con mu noto, e alle numerosita' che si avranno davvero.

In [ ]:
r = np.random.default_rng(RANDOM_SEED)
print("Hill su leggi di potenza sintetiche (mu vero -> mu stimato)\n")
print(f"  {'mu vero':>8s} {'n':>8s} {'mu stimato':>22s} {'errore':>9s}")
for mu_vero in (2.0, 2.4, 3.0, 3.5):
    a_vero = mu_vero - 1.0
    for n in (500, 2000, 12000):
        # Pareto con survival P(X>x) = x^(-a) per x >= 1
        x = (1.0 - r.random(n)) ** (-1.0 / a_vero)
        res = hill_boot(x, r, nboot=120)
        print(f"  {mu_vero:>8.1f} {n:>8,} "
              f"{res['mu']:>10.3f} [{res['lo']:.3f}, {res['hi']:.3f}] "
              f"{res['mu']-mu_vero:>+9.3f}")
    print()

print("controllo negativo: su una esponenziale (nessuna coda a potenza)")
x = r.exponential(1000, 12000)
res = hill_boot(x, r, nboot=120)
print(f"  mu stimato = {res['mu']:.2f} [{res['lo']:.2f}, {res['hi']:.2f}]  "
      f"-> molto sopra 3, come deve essere (varianza finita)")

## 4. I quattro corpora

In [ ]:
GUT_START = re.compile(r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
GUT_END   = re.compile(r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK.*?\*\*\*", re.S)
STRUCT = re.compile(r"^[ \t]*(?:BOOK\s+[A-Z]+[^\n]*|CHAPTER\s+[IVXLCDM\d]+[^\n]*|"
                    r"(?:FIRST|SECOND)\s+EPILOGUE[^\n]*|EPILOGUE[^\n]*|CONTENTS[^\n]*|"
                    r"PART\s+[IVXLCDM\d]+[^\n]*|APPENDIX[^\n]*|\d+)[ \t]*$", re.M)
APPARATO = re.compile(r"^==+\s*(See also|References|Notes|Citations|Sources|Bibliography|"
                      r"Further reading|External links|Works cited|Footnotes|"
                      r"Explanatory notes|General sources)\s*==+\s*$", re.M | re.I)
INTEST = re.compile(r"^==+.*?==+\s*$", re.M)
TTS = re.compile(r'<span[^>]*data-tts-block="true"[^>]*>(.*?)</span>', re.S)
ARTICOLO = re.compile(r"<article[^>]*>(.*?)</article>", re.S)
WORD_CHAR = r"[^\W\d_]"
TOKEN_RE  = re.compile(WORD_CHAR + r"+(?:['\u2019\-]" + WORD_CHAR + r"+)*", re.UNICODE)
SENT_END  = set(".!?")

def contesto_ssl():
    if HAS_CERTIFI:
        try: return ssl.create_default_context(cafile=certifi.where())
        except Exception: pass
    try: return ssl.create_default_context()
    except ssl.SSLError:
        c = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        c.check_hostname = False; c.verify_mode = ssl.CERT_NONE
        return c
CTX = contesto_ssl()

def _frag(h):
    h = re.sub(r"(?is)<(script|style|button|svg|nav|footer|aside)[^>]*>.*?</\1>", " ", h)
    h = re.sub(r"<[^>]+>", " ", h); h = htmlmod.unescape(h)
    h = re.sub(r"\[\d+\]", " ", h)
    h = re.sub(r"[ \t\u00a0]+", " ", h)
    return re.sub(r"\n\s*\n+", "\n\n", h).strip()

def safe(t): return re.sub(r"[^A-Za-z0-9_]", "", t.replace(" ", "_"))[:60]

def carica_wiki(t):
    fn = CACHE_WIKI / (hashlib.sha256(t.encode()).hexdigest()[:14] + ".txt")
    if not fn.exists(): return ""
    tx = fn.read_bytes().decode("utf-8", errors="replace")
    m = APPARATO.search(tx)
    if m: tx = tx[:m.start()]
    tx = INTEST.sub("", tx)
    tx = re.sub(r"\n[ \t]+\n", "\n\n", tx)
    return re.sub(r"\n{3,}", "\n\n", tx).strip()

def carica_grok(t):
    fn = CACHE_GROK / (safe(t) + ".html")
    if not fn.exists(): return ""
    p = fn.read_bytes().decode("utf-8", errors="replace")
    b = [x for x in (_frag(y) for y in TTS.findall(p)) if len(x) > 40]
    if b: return "\n\n".join(b)
    m = ARTICOLO.search(p)
    return _frag(m.group(1)) if m else ""

def carica_v01(t):
    fn = CORPUS_V01 / (safe(t) + ".txt")
    return fn.read_bytes().decode("utf-8", errors="replace") if fn.exists() else ""

def libro():
    fn = CACHE_LIB / (hashlib.sha256(GUTENBERG_URL.encode()).hexdigest()[:12] + ".txt")
    if fn.exists():
        raw = fn.read_bytes().decode("utf-8", errors="replace")
        if "\r\r" not in raw: return raw
    req = urllib.request.Request(GUTENBERG_URL, headers={"User-Agent": "ricerca/1.0"})
    with urllib.request.urlopen(req, timeout=90, context=CTX) as rr:
        raw = rr.read().decode("utf-8", errors="replace")
    CACHE_LIB.mkdir(parents=True, exist_ok=True)
    fn.write_bytes(raw.encode("utf-8"))
    return raw

raw = libro().replace("\r\n", "\n").replace("\r", "\n")
raw = raw[GUT_START.search(raw).end():]
raw = raw[:GUT_END.search(raw).start()]
raw = raw[raw.find(START_PHRASE):]
raw = STRUCT.sub("", raw)
raw = re.sub(r"\n[ \t]+\n", "\n\n", raw)
BODY = re.sub(r"\n{3,}", "\n\n", raw).strip()[PROMPT_CHARS:]

R = pd.read_csv(RIEPILOGO)
TITOLI = [t for t in R["titolo"] if t not in ESCLUSI]
CORPORA = {}
for nome, f in [("wikipedia", carica_wiki), ("grok_v01", carica_v01),
                ("grok_oggi", carica_grok)]:
    CORPORA[nome] = {t: f(t) for t in TITOLI}
TITOLI = [t for t in TITOLI if all(len(CORPORA[c].get(t, "")) >= N_EFF for c in CORPORA)]
for c in list(CORPORA):
    CORPORA[c] = {t: CORPORA[c][t] for t in TITOLI}
starts = np.linspace(0, max(len(BODY) - N_EFF, 0), N_SEG_LETT).astype(int)
CORPORA["letterario"] = {f"wrnpc_{i}": BODY[s:s+N_EFF] for i, s in enumerate(starts)}
print(f"corpo letterario: {len(BODY):,} caratteri")
print(f"terne complete: {len(TITOLI)} | corpora: {list(CORPORA)}")

In [ ]:
STOPWORDS = set("""a about above after again against all am an and any are as at be because been
before being below between both but by can cannot could did do does doing down during each few for
from further had has have having he her here hers herself him himself his how i if in into is it its
itself me more most my myself no nor not of off on once only or other ought our ours ourselves out
over own same she should so some such than that the their theirs them themselves then there these
they this those through to too under until up very was we were what when where which while who whom
why with would you your yours yourself yourselves said one two would shall may might must upon""".split())
VOWELS = set("aeiou")
_cre = {}
def pos_parola(text, w):
    if w not in _cre:
        _cre[w] = re.compile(r"(?<!" + WORD_CHAR + r")" + re.escape(w) +
                             r"(?!" + WORD_CHAR + r")", re.IGNORECASE | re.UNICODE)
    return np.fromiter((m.start() for m in _cre[w].finditer(text)), dtype=np.int64)

def word_stats(text):
    freq, cap, tot = Counter(), Counter(), Counter()
    pe, primo = 0, True
    for m in TOKEN_RE.finditer(text):
        w = m.group(0); lw = w.lower(); freq[lw] += 1
        gap = text[pe:m.start()]
        if not (primo or any(c in SENT_END for c in gap) or "\n\n" in gap):
            tot[lw] += 1
            if w[0].isupper(): cap[lw] += 1
        pe, primo = m.end(), False
    return freq, {w: (cap[w]/tot[w] if tot[w] >= 3 else 0.0) for w in freq}

def select_targets(text):
    low = text.lower()
    letters = [c for c, _ in Counter(c for c in low if c.isalpha() and c.isascii())
               .most_common(N_LETTERS)]
    freq, capr = word_stats(text)
    ordered = [w for w, _ in freq.most_common() if len(w) >= 2]
    funcs = [w for w in ordered if w in STOPWORDS][:N_FUNCTION]
    isp = lambda w: capr.get(w, 0) >= PROPER_CAP_RATIO
    isk = lambda w: isp(w) or (w not in STOPWORDS and len(w) >= 4)
    keys = [w for w in ordered if isk(w)][:N_KEYWORDS]
    ks = set(keys)
    pool = [w for w in ordered if w not in ks and not isp(w) and w not in funcs]
    matched, used = [], set()
    for k in keys[:N_MATCHED]:
        fk = freq[k]
        cand = sorted((w for w in pool if w not in used),
                      key=lambda w: (abs(math.log((freq[w]+1e-9)/(fk+1e-9))), w))
        if cand: matched.append(cand[0]); used.add(cand[0])
    return ([("vocali", VOWELS, "vocali"), ("spazio", " ", "spazio")] +
            [(c, c, "lettera") for c in letters] +
            [(w, w, "funzione") for w in funcs] +
            [(w, w, "keyword") for w in keys] +
            [(w, w, "appaiata") for w in matched])

# --- raccolta degli intervalli, normalizzati per la media di ciascuna sequenza ---
POOL = {}     # (corpus, tipo) -> lista di array di tau normalizzati
DETT = []
for corpus, docs in CORPORA.items():
    print(f"{corpus:12s} ", end="", flush=True)
    for lab, testo in docs.items():
        t = testo[:N_EFF]
        carr = np.array(list(t.lower()))
        for etichetta, chiave, tipo in select_targets(t):
            if tipo == "vocali":  pos = np.flatnonzero(np.isin(carr, list(chiave)))
            elif tipo == "spazio": pos = np.flatnonzero(carr == " ")
            elif tipo == "lettera": pos = np.flatnonzero(carr == chiave)
            else: pos = pos_parola(t, chiave)
            if len(pos) < MIN_EVENTS: continue
            tau = np.diff(pos).astype(float)
            if len(tau) < MIN_TAU_SEQ: continue
            POOL.setdefault((corpus, tipo), []).append(tau / tau.mean())
            DETT.append(dict(corpus=corpus, titolo=lab, sequenza=etichetta, tipo=tipo,
                             n_tau=len(tau), mean_tau=float(tau.mean())))
        print(".", end="", flush=True)
    print(" fatto")
D = pd.DataFrame(DETT)
print(f"\nsequenze nel pool: {len(D):,} | intervalli totali: {int(D['n_tau'].sum()):,}")
print(D.groupby(["corpus", "tipo"])["n_tau"].agg(["size", "sum"]).to_string())

## 5. Stima di mu per corpus e livello

In [ ]:
r = np.random.default_rng(RANDOM_SEED)
COL = {"letterario": "#333333", "wikipedia": "#0072B2",
       "grok_v01": "#E69F00", "grok_oggi": "#D55E00"}
ORD = ["letterario", "wikipedia", "grok_v01", "grok_oggi"]
NOMI = {"letterario": "Guerra e pace", "wikipedia": "Wikipedia",
        "grok_v01": "Grokipedia v0.1", "grok_oggi": "Grokipedia oggi"}
LIV = [("lettera", "lettere"), ("funzione", "parole funzione"),
       ("appaiata", "controlli appaiati"), ("keyword", "keyword")]

righe = []
for (corpus, tipo), lst in POOL.items():
    x = np.concatenate(lst)
    res = hill_boot(x, r)
    res.update(corpus=corpus, tipo=tipo, n_seq=len(lst))
    righe.append(res)
MU = pd.DataFrame(righe)
MU["banda_2_3"] = (MU["mu"] > 2) & (MU["mu"] < 3)
MU["gamma_pred"] = 4.0 - MU["mu"]
MU.to_csv(OUT_DIR / "dati" / "mu_hill.csv", index=False)

print("Esponente di coda mu, stimato per pool (Hill, IC 95% bootstrap)")
print("La banda 2 < mu < 3 e' quella in cui l'Eq. 8 vale e la varianza diverge.\n")
print(f"  {'livello':20s}" + "".join(f"{NOMI[c]:>24s}" for c in ORD))
for tipo, nome in LIV:
    riga = f"  {nome:20s}"
    for c in ORD:
        s = MU[(MU.corpus == c) & (MU.tipo == tipo)]
        if len(s):
            x = s.iloc[0]
            flag = "*" if x["banda_2_3"] else " "
            riga += f"{x['mu']:>13.2f} [{x['lo']:.2f},{x['hi']:.2f}]{flag}"
        else:
            riga += f"{'n.d.':>24s}"
    print(riga)
print("\n  * = dentro la banda 2 < mu < 3 prevista dall'Eq. 8")
print(f"\n  intervalli usati per stima: {MU['n'].min():,} - {MU['n'].max():,}"
      f" | k di Hill: {MU['k'].min():,} - {MU['k'].max():,}")

## 6. Il test dell'Eq. 8

Il confronto e' fra 4 - mu (predetto) e gamma_A2 (osservato), perche' A2 permuta gli
intervalli e realizza quindi l'ipotesi C_tau(k) = delta(k) dell'Eq. 8. Si riporta anche
gamma originale, che deve stare **sopra** gamma_A2: e' la disuguaglianza gamma >= gamma_A2
del paper.

### Diagnostica: il plateau di Hill

In [ ]:
# --- il plateau di Hill esiste? Senza, la stima non e' interpretabile ---
# Lo stimatore restituisce un numero per qualunque campione, anche se la coda NON e'
# una legge di potenza. L'unico controllo e' guardare alpha in funzione di k: se c'e'
# una legge di potenza si vede un tratto piatto, altrimenti la curva deriva.
MU_EXP = 4.08   # valore che Hill restituisce su una esponenziale pura
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
KFRAZ = np.logspace(np.log10(0.005), np.log10(0.35), 26)

ax = axes[0]
r2 = np.random.default_rng(7)
for mu_v, col in [(2.4, "#0072B2"), (3.0, "#009E73")]:
    x = (1.0 - r2.random(20000)) ** (-1.0 / (mu_v - 1))
    ax.plot(KFRAZ, [hill_mu(x, fraz=f)[0] for f in KFRAZ], "-", color=col,
            label=f"legge di potenza mu={mu_v}")
    ax.axhline(mu_v, color=col, ls=":", lw=1)
x = r2.exponential(1000, 20000)
ax.plot(KFRAZ, [hill_mu(x, fraz=f)[0] for f in KFRAZ], "-", color="crimson",
        label="esponenziale (nessuna coda)")
ax.set_xscale("log"); ax.set_xlabel("frazione di code usata (k/n)")
ax.set_ylabel("mu stimato"); ax.legend(fontsize=7)
ax.set_title("A) riferimento: come si legge un Hill plot", fontsize=10)

for j, tipo in enumerate(["keyword", "lettera"]):
    ax = axes[j + 1]
    for c_ in ORD:
        lst = POOL.get((c_, tipo))
        if not lst: continue
        x = np.concatenate(lst)
        ax.plot(KFRAZ, [hill_mu(x, fraz=f)[0] for f in KFRAZ], "-", lw=1.8,
                color=COL[c_], label=NOMI[c_])
    ax.axhspan(2, 3, color="green", alpha=.08)
    ax.axhline(MU_EXP, color="crimson", ls="--", lw=1.2)
    ax.annotate("esponenziale", (KFRAZ[-1], MU_EXP), fontsize=7.5, color="crimson",
                ha="right", va="bottom")
    ax.set_xscale("log"); ax.set_xlabel("frazione di code usata (k/n)")
    ax.set_ylabel("mu stimato"); ax.legend(fontsize=7)
    ax.set_title(f"{'BC'[j]}) {tipo}", fontsize=10)
fig.suptitle("Diagnostica dello stimatore di Hill: il plateau c'e'?", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "hill_plot.png"); plt.show()

print("Sensibilita' di mu (keyword) alla frazione di code usata:")
for c_ in ORD:
    lst = POOL.get((c_, "keyword"))
    if not lst: continue
    x = np.concatenate(lst)
    v = [hill_mu(x, fraz=f)[0] for f in (0.02, 0.05, 0.10, 0.20)]
    print(f"  {NOMI[c_]:17s} k/n=2%: {v[0]:.2f}   5%: {v[1]:.2f}   "
          f"10%: {v[2]:.2f}   20%: {v[3]:.2f}   escursione {max(v)-min(v):.2f}")

## 6. Il modello a potenza troncata

In [ ]:
# --- il modello che il paper assume davvero: potenza CON taglio esponenziale ---
#     p(tau) ~ tau^(-mu) * exp(-tau/tau_c)   su tau >= xmin
# Il Hill plot (cella precedente) mostra che non esiste plateau: la coda non e' una
# potenza pura, e Hill --- che quella assume --- restituisce un valore che dipende da
# quanta coda si usa. Il modello troncato e' quello che Altmann postula quando dice che
# scendendo nella gerarchia i vuoti lunghi vengono spezzati, introducendo un cut-off.
# Qui mu e tau_c vengono STIMATI insieme invece che assunti.

def _Z(mu, xc, xmin):
    v, _ = integrate.quad(lambda t: t**(-mu)*math.exp(-t/xc), xmin, np.inf, limit=200)
    return v

def nll_tronc(par, x, xmin):
    mu, lxc = par
    xc = math.exp(lxc)
    if mu <= -1 or mu > 8 or xc <= 1e-3 or xc > 1e6: return 1e12
    Z = _Z(mu, xc, xmin)
    if not np.isfinite(Z) or Z <= 0: return 1e12
    return -(-mu*np.sum(np.log(x)) - np.sum(x)/xc - len(x)*math.log(Z))

def nll_pareto(mu, x, xmin):
    if mu <= 1.001: return 1e12
    return -(len(x)*math.log(mu-1) + (mu-1)*len(x)*math.log(xmin) - mu*np.sum(np.log(x)))

def nll_lnorm(par, x, xmin):
    m, s = par
    if s <= 0.01: return 1e12
    z = (np.log(x) - m)/s
    lp = -np.log(x*s*math.sqrt(2*math.pi)) - z*z/2
    coda = 0.5*math.erfc((math.log(xmin)-m)/(s*math.sqrt(2)))
    if coda <= 0: return 1e12
    return -(np.sum(lp) - len(x)*math.log(coda))

def fit_tronc(x, xmin, start=None):
    s = start or [2.5, math.log(max(float(x.mean()), xmin*2))]
    r = optimize.minimize(nll_tronc, s, args=(x, xmin), method="Nelder-Mead",
                          options=dict(maxiter=1200, xatol=1e-5, fatol=1e-5))
    return float(r.x[0]), float(math.exp(r.x[1])), float(r.fun)

def profilo_mu(mu, x, xmin):
    """per mu fissato, il miglior tau_c e la verosimiglianza corrispondente"""
    f = lambda l: nll_tronc([mu, l[0]], x, xmin)
    r = optimize.minimize(f, [math.log(max(float(x.mean()), xmin*2))],
                          method="Nelder-Mead", options=dict(maxiter=600, xatol=1e-5))
    return float(r.fun), float(math.exp(r.x[0]))

def ic_profilo(x, xmin, mu_h, f_h, soglia=3.84):
    """IC 95% da profilo di verosimiglianza, per bisezione. Piu' affidabile
    dell'errore asintotico perche' mu e tau_c sono fortemente correlati."""
    out = []
    for direzione in (-1, +1):
        lo, hi = mu_h, mu_h
        for _ in range(40):
            hi = hi + direzione*0.1
            if hi < -0.5 or hi > 6: break
            if 2*(profilo_mu(hi, x, xmin)[0] - f_h) > soglia: break
        a, b = lo, hi
        for _ in range(30):
            m = (a + b)/2
            if 2*(profilo_mu(m, x, xmin)[0] - f_h) > soglia: b = m
            else: a = m
        out.append(a)
    return min(out), max(out)

# validazione: su una potenza troncata sintetica si recuperano i parametri veri?
r_ = np.random.default_rng(RANDOM_SEED)
print("Validazione del fit troncato su dati sintetici (mu, tau_c veri -> stimati)\n")
for mu_v, xc_v in [(2.4, 10.0), (2.4, 30.0), (3.0, 10.0)]:
    # campionamento per rigetto da una Pareto con peso exp(-t/xc)
    # campionamento per rigetto, con guardia esplicita sul numero di tentativi
    acc = np.empty(0)
    for _tent in range(200):
        if len(acc) >= 4000:
            break
        t = (1.0 - r_.random(20000)) ** (-1.0/(mu_v - 1))
        acc = np.concatenate([acc, t[r_.random(len(t)) < np.exp(-t/xc_v)]])
    else:
        print(f"  ATTENZIONE: campionamento non convergente per mu={mu_v}, xc={xc_v}")
        continue
    x = acc[:4000]
    xmin = float(np.percentile(x, 90)); xt = x[x >= xmin]
    m_, c_, _ = fit_tronc(xt, xmin)
    print(f"  mu={mu_v:.1f} tau_c={xc_v:>4.0f}  ->  mu={m_:.2f} tau_c={c_:.1f}"
          f"   (n coda = {len(xt):,})")

In [ ]:
# --- stima di (mu, tau_c) su ogni pool, con IC di profilo e confronto fra modelli ---
QUANT_XMIN = 90          # xmin = percentile di questo ordine
N_BOOT_T   = 80

r_ = np.random.default_rng(RANDOM_SEED)
righe = []
for (corpus, tipo), lst in POOL.items():
    x0 = np.concatenate(lst)
    xmin = float(np.percentile(x0, QUANT_XMIN))
    x = x0[x0 >= xmin]
    if len(x) < 150: continue
    mu_h, xc_h, f_h = fit_tronc(x, xmin)
    lo, hi = ic_profilo(x, xmin, mu_h, f_h)
    # confronto: potenza pura (annidata) e log-normale (non annidata)
    mu_p = 1 + len(x)/np.sum(np.log(x/xmin))
    f_p = nll_pareto(mu_p, x, xmin)
    LR = 2*(f_p - f_h)
    p_lr = sps.chi2.sf(max(LR, 0.0), df=1)
    r_ln = optimize.minimize(nll_lnorm, [np.log(x).mean(), np.log(x).std()],
                             args=(x, xmin), method="Nelder-Mead",
                             options=dict(maxiter=800))
    righe.append(dict(corpus=corpus, tipo=tipo, n_coda=len(x), xmin=xmin,
                      mu=mu_h, mu_lo=lo, mu_hi=hi, tau_c=xc_h,
                      mu_hill=mu_p, LR_vs_pura=LR, p_pura=p_lr,
                      meglio_lognorm=bool(r_ln.fun < f_h - 2)))
MU = pd.DataFrame(righe)
MU["banda_2_3"] = (MU["mu"] > 2) & (MU["mu"] < 3)
MU["tetto_gamma"] = 4.0 - MU["mu"]
MU.to_csv(OUT_DIR / "dati" / "coda_troncata.csv", index=False)

print(f"Stima congiunta di (mu, tau_c), xmin = percentile {QUANT_XMIN}\n")
print(f"  {'corpus':17s}{'livello':20s}{'n coda':>8s}{'mu':>21s}{'tau_c':>8s}"
      f"{'mu Hill':>9s}{'p pura':>9s}")
for c_ in ORD:
    for tipo, nome in LIV:
        s = MU[(MU.corpus == c_) & (MU.tipo == tipo)]
        if not len(s): continue
        x_ = s.iloc[0]
        ln = "  log-normale meglio" if x_["meglio_lognorm"] else ""
        print(f"  {NOMI[c_]:17s}{nome:20s}{int(x_['n_coda']):>8,}"
              f"{x_['mu']:>10.2f} [{x_['mu_lo']:.2f},{x_['mu_hi']:.2f}]"
              f"{x_['tau_c']:>8.1f}{x_['mu_hill']:>9.2f}{x_['p_pura']:>9.1e}{ln}")
    print()
print("p pura = test del rapporto di verosimiglianza contro la potenza SENZA taglio.")
print("p piccolo -> il taglio serve, la potenza pura e' rifiutata.")
print(f"  celle in cui la potenza pura e' rifiutata: "
      f"{int((MU['p_pura'] < 0.05).sum())}/{len(MU)}")

# degenerazione fra i due parametri
print("\nmu e tau_c sono identificabili separatamente?")
for c_ in ORD:
    s = MU[(MU.corpus == c_) & (MU.tipo == "keyword")]
    if not len(s): continue
    x0 = np.concatenate(POOL[(c_, "keyword")])
    xmin = float(np.percentile(x0, QUANT_XMIN)); x = x0[x0 >= xmin]
    mus, xcs = [], []
    for _ in range(N_BOOT_T):
        xb = x[r_.integers(0, len(x), len(x))]
        m_, c2_, _ = fit_tronc(xb, xmin, start=[s["mu"].iloc[0], math.log(s["tau_c"].iloc[0])])
        if np.isfinite(m_) and 0 < c2_ < 1e5:
            mus.append(m_); xcs.append(c2_)
    rr, _ = sps.pearsonr(mus, np.log(xcs))
    print(f"  {NOMI[c_]:17s} keyword: corr(mu, ln tau_c) = {rr:+.3f}  "
          f"-> {'FORTEMENTE degeneri' if abs(rr) > .8 else 'separabili'}")
print("\n  Conseguenza: mu non va mai riportato da solo, sempre la coppia (mu, tau_c)")
print("  oppure l'intervallo di profilo, che tiene conto della correlazione.")

In [ ]:
# --- l'Eq. 8 nella forma corretta in presenza di taglio: una DISUGUAGLIANZA ---
# Altmann deriva gamma = 4 - mu per la potenza PURA. Con un taglio a tau_c il paper
# stesso osserva che la burstiness contribuisce a gamma solo per t < tau_c: il gamma
# effettivo e' quindi MINORE di 4 - mu. La previsione verificabile e'
#         gamma_A2 <= 4 - mu
# con l'ulteriore vincolo fisico gamma <= 2 (diffusione balistica).
A = pd.read_csv(SEQUENZE, low_memory=False)
A = A[A["stimabile"] == True]
oss = A.groupby(["corpus", "tipo"])[["gamma", "gamma_A2", "gamma_A1"]].agg(["mean", "std"])
oss.columns = ["_".join(x) for x in oss.columns]
T = MU.merge(oss.reset_index(), on=["corpus", "tipo"], how="left")
T["margine"] = T["tetto_gamma"] - T["gamma_A2_mean"]
T["rispetta"] = T["margine"] >= 0
T.to_csv(OUT_DIR / "dati" / "test_eq8.csv", index=False)

print("EQ. 8 COME DISUGUAGLIANZA:   gamma_A2 <= 4 - mu\n")
print(f"  {'corpus':17s}{'livello':20s}{'mu':>7s}{'tau_c':>8s}{'tetto':>8s}"
      f"{'gamma_A2':>10s}{'margine':>9s}{'':>4s}")
for c_ in ORD:
    for tipo, nome in LIV:
        s = T[(T.corpus == c_) & (T.tipo == tipo)]
        if not len(s): continue
        x_ = s.iloc[0]
        ok = "  ok" if x_["rispetta"] else "  VIOLATA"
        print(f"  {NOMI[c_]:17s}{nome:20s}{x_['mu']:>7.2f}{x_['tau_c']:>8.1f}"
              f"{x_['tetto_gamma']:>8.2f}{x_['gamma_A2_mean']:>10.3f}"
              f"{x_['margine']:>+9.3f}{ok}")
    print()
print(f"disuguaglianza rispettata: {int(T['rispetta'].sum())}/{len(T)} celle")
print(f"  margine mediano {T['margine'].median():+.3f}  "
      f"[{T['margine'].min():+.3f}, {T['margine'].max():+.3f}]")
print(f"gamma >= gamma_A2 (disuguaglianza del paper): "
      f"{int((T['gamma_mean'] >= T['gamma_A2_mean']).sum())}/{len(T)} celle")

# nella banda 2<mu<3 e senza taglio, la previsione tornerebbe un'uguaglianza
ban = T[T["banda_2_3"]]
print(f"\ncelle dentro la banda 2 < mu < 3: {len(ban)}/{len(T)}")
if len(ban):
    print(f"  margine medio in banda: {ban['margine'].mean():+.3f}  "
          f"(quanto il taglio comprime gamma sotto il tetto teorico)")

## 8. Il taglio e' reale o e' la finestra?

In [ ]:
# --- tau_c e' reale o e' il nostro tetto? ------------------------------------
# Con una finestra da N caratteri nessun tau puo' superare N, quindi normalizzando
# per la media il tetto e' N/<tau>. Se tau_c stimato crescesse proporzionalmente
# alla finestra sarebbe un artefatto; se si stabilizza e' una proprieta' del testo.
righe = []
for N in [30_000, 60_000, 120_000, 240_000, 480_000]:
    if N > len(BODY): break
    n_seg = max(6, len(BODY) // N)
    acc = []
    for s in np.linspace(0, len(BODY) - N, min(n_seg, 20)).astype(int):
        seg = BODY[s:s+N]
        for etichetta, chiave, tipo in select_targets(seg):
            if tipo != "keyword": continue
            pos = pos_parola(seg, chiave)
            if len(pos) < MIN_EVENTS: continue
            tau = np.diff(pos).astype(float)
            if len(tau) < MIN_TAU_SEQ: continue
            acc.append(tau / tau.mean())
    if not acc: continue
    x0 = np.concatenate(acc)
    xmin = float(np.percentile(x0, QUANT_XMIN)); x = x0[x0 >= xmin]
    m_, c_, _ = fit_tronc(x, xmin)
    righe.append(dict(N=N, n_tau=len(x0), max_norm=float(x0.max()),
                      mu=m_, tau_c=c_))
    print(f"  N = {N:>8,}  n tau = {len(x0):>7,}  max normalizzato = {x0.max():>6.1f}"
          f"   mu = {m_:.2f}   tau_c = {c_:.1f}")
FIN_DF = pd.DataFrame(righe)
FIN_DF.to_csv(OUT_DIR / "dati" / "tau_c_vs_finestra.csv", index=False)
if len(FIN_DF) >= 3:
    rap = FIN_DF["tau_c"].iloc[-1] / FIN_DF["tau_c"].iloc[0]
    rapN = FIN_DF["N"].iloc[-1] / FIN_DF["N"].iloc[0]
    print(f"\n  la finestra e' cresciuta di un fattore {rapN:.0f}, tau_c di {rap:.2f}")
    print("  -> " + ("tau_c segue la finestra: il taglio e' in larga parte un ARTEFATTO"
                     if rap > 0.5*rapN else
                     "tau_c NON segue la finestra: il taglio e' una proprieta' del testo"))

## 7. Controllo sul dato del paper

Il paper stima mu = 2.4 per Guerra e pace e riporta gamma = 1.68 per *prince*. Qui si
stima mu direttamente sugli intervalli di *prince* **sul libro intero**, dove il campione
e' grande abbastanza per una stima di coda seria.

In [ ]:
r = np.random.default_rng(RANDOM_SEED)
print("mu stimato sugli intervalli del libro intero\n")
print(f"  {'parola':12s}{'occorrenze':>11s}{'intervalli':>11s}{'mu':>21s}"
      f"{'4-mu':>8s}{'cv_tau':>9s}")
righe = []
for w in ["prince", "pierre", "andrew", "princess", "e", "t"]:
    if len(w) == 1:
        carr = np.array(list(BODY.lower()))
        pos = np.flatnonzero(carr == w)
    else:
        pos = pos_parola(BODY, w)
    if len(pos) < 60: continue
    tau = np.diff(pos).astype(float)
    res = hill_boot(tau, r)
    cv = tau.std(ddof=1) / tau.mean()
    print(f"  {w:12s}{len(pos):>11,}{len(tau):>11,}"
          f"{res['mu']:>10.2f} [{res['lo']:.2f},{res['hi']:.2f}]"
          f"{4-res['mu']:>8.2f}{cv:>9.2f}")
    righe.append(dict(parola=w, n=len(pos), mu=res["mu"], lo=res["lo"], hi=res["hi"],
                      cv_tau=cv))
LIBRO = pd.DataFrame(righe)
LIBRO.to_csv(OUT_DIR / "dati" / "mu_libro_intero.csv", index=False)
pr = LIBRO[LIBRO.parola == "prince"]
if len(pr):
    x = pr.iloc[0]
    print(f"\n  confronto col paper (Altmann et al. 2012):")
    print(f"    mu       stimato {x['mu']:.2f} [{x['lo']:.2f}, {x['hi']:.2f}]"
          f"   contro  2.4 assunto nel paper")
    print(f"    4 - mu   = {4-x['mu']:.2f}                    contro  gamma = 1.68 misurato")
    print(f"    cv_tau   = {x['cv_tau']:.2f}                    contro  3.86 del paper")

## 8. Figure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.7))

xs = np.arange(len(LIV))
ax = axes[0]
# Dove tau_c ~ 1 il modello troncato si riduce a una esponenziale pura: mu non e' piu'
# un esponente di coda ma un parametro di forma, e leggerlo sulla scala della banda
# 2<mu<3 induce la conclusione OPPOSTA a quella vera (le lettere non hanno coda a
# potenza affatto). Quei punti si marcano come non interpretabili.
TAUC_MIN = 3.0
for i, c in enumerate(ORD):
    v, lo, hi, deg = [], [], [], []
    for tipo, _ in LIV:
        s_ = MU[(MU.corpus == c) & (MU.tipo == tipo)]
        v.append(s_["mu"].iloc[0] if len(s_) else np.nan)
        lo.append(s_["mu_lo"].iloc[0] if len(s_) else np.nan)
        hi.append(s_["mu_hi"].iloc[0] if len(s_) else np.nan)
        deg.append(bool(len(s_)) and s_["tau_c"].iloc[0] < TAUC_MIN)
    v, lo, hi = np.array(v), np.array(lo), np.array(hi)
    err = np.vstack([np.clip(v-lo, 0, None), np.clip(hi-v, 0, None)])
    x = xs + .06*(i-1.5)
    ax.errorbar(x, v, yerr=err, color=COL[c], lw=1.5, ls="-", marker="none",
                capsize=3, label=NOMI[c], alpha=.9)
    d = np.array(deg)
    ax.scatter(x[~d], v[~d], s=70, marker="os^D"[i], color=COL[c],
               edgecolors="k", linewidths=.6, zorder=3)
    ax.scatter(x[d], v[d], s=70, marker="os^D"[i], facecolors="white",
               edgecolors=COL[c], linewidths=1.4, zorder=3)
ax.axhspan(2, 3, color="green", alpha=.08)
ax.axhline(2.4, color="crimson", ls="--", lw=1.4)
ax.annotate("$\\mu$ = 2.4 (paper)", (len(LIV)-.55, 2.42), fontsize=7.5,
            color="crimson", va="bottom", ha="right")
ax.scatter([], [], s=70, marker="o", facecolors="white", edgecolors="grey",
           linewidths=1.4, label="fit degenerato in\nesponenziale ($\\tau_c<3$)")
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel("$\\mu$ (modello troncato)")
ax.set_title("A) esponente di coda\n"
             "banda verde: varianza divergente — simboli vuoti: $\\mu$ non interpretabile",
             fontsize=9.5)
ax.legend(fontsize=6.5, loc="lower right")

ax = axes[1]
# x = mu, y = gamma_A2, con sopra la curva teorica a tratti. Mettere in ascissa la
# previsione invece di mu produrrebbe 23 punti sovrapposti a x = 1 e uno isolato:
# una figura che sembra un test e non lo e'.
mm = np.linspace(2.02, 4.4, 400)
gt = np.where(mm >= 3.0, 1.0, 4.0 - mm)
ax.plot(mm, 4.0 - mm, "k--", lw=1.6, zorder=1, label="tetto 4 - mu")
ax.fill_between(mm, 1.0, np.maximum(4.0-mm, 1.0), color="grey", alpha=.10)
ax.axvspan(2, 3, color="green", alpha=.08)
MK = {"lettera": "o", "funzione": "s", "appaiata": "^", "keyword": "D"}
for c_ in ORD:
    for tipo, nome in LIV:
        s_ = T[(T.corpus == c_) & (T.tipo == tipo)]
        if not len(s_): continue
        x_ = s_.iloc[0]
        ax.scatter(x_["mu"], x_["gamma_A2_mean"], s=80, color=COL[c_],
                   marker=MK[tipo], edgecolors="k", linewidths=.6, zorder=3)
for tipo, nome in LIV:
    ax.scatter([], [], s=70, marker=MK[tipo], color="grey", edgecolors="k",
               linewidths=.6, label=nome)
ax.set_xlabel("$\\mu$ (modello troncato)")
ax.set_ylabel("gamma_A2 (osservato)")
ax.set_title("B) gamma_A2 sotto il tetto 4 - mu\ncolore = corpus, forma = livello",
             fontsize=10)
ax.legend(fontsize=6.5, loc="upper right")

ax = axes[2]
for c in ORD:
    s = T[T.corpus == c].set_index("tipo").reindex([t for t, _ in LIV])
    ax.plot(xs, s["gamma_mean"], "o-", color=COL[c], ms=6, lw=1.6, label=NOMI[c])
    ax.plot(xs, s["gamma_A2_mean"], "s--", color=COL[c], ms=5, lw=1.2, alpha=.6)
ax.axhline(1, color="grey", ls=":", lw=1.2)
ax.set_xticks(xs); ax.set_xticklabels([n for _, n in LIV], rotation=12, fontsize=8)
ax.set_ylabel("gamma"); ax.set_title("C) gamma (pieno) e gamma_A2 (tratteggio)\n"
                                     "il paper prevede gamma >= gamma_A2", fontsize=10)
ax.legend(fontsize=7)

fig.suptitle(f"Esponente di coda e test dell'Eq. 8 di Altmann  "
             f"(N = {N_EFF:,} caratteri, {len(TITOLI)} terne)", fontweight="bold")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure" / "esponente_coda.png"); plt.show()

## 9. Sintesi

In [ ]:
L = []
L.append("CODA DEGLI INTERVALLI: POTENZA TRONCATA E TEST DELL'EQ. 8")
L.append("=" * 68)
L.append(f"terne: {len(TITOLI)} | N_EFF = {N_EFF:,} caratteri")
L.append("")
L.append("MODELLO:  p(tau) ~ tau^(-mu) * exp(-tau/tau_c)")
L.append(f"  potenza pura rifiutata in {int((MU['p_pura'] < 0.05).sum())}/{len(MU)} celle "
         f"(test del rapporto di verosimiglianza)")
L.append("")
L.append("KEYWORD — (mu, tau_c) con IC 95% di profilo")
for c_ in ORD:
    s = MU[(MU.corpus == c_) & (MU.tipo == "keyword")]
    if not len(s): continue
    x_ = s.iloc[0]
    L.append(f"  {NOMI[c_]:17s} mu = {x_['mu']:.2f} [{x_['mu_lo']:.2f}, {x_['mu_hi']:.2f}]"
             f"   tau_c = {x_['tau_c']:.1f}   (Hill dava {x_['mu_hill']:.2f})")
L.append("")
L.append("EQ. 8 COME DISUGUAGLIANZA  gamma_A2 <= 4 - mu")
L.append(f"  rispettata in {int(T['rispetta'].sum())}/{len(T)} celle, "
         f"margine mediano {T['margine'].median():+.3f}")
L.append(f"  gamma >= gamma_A2 (disuguaglianza del paper): "
         f"{int((T['gamma_mean'] >= T['gamma_A2_mean']).sum())}/{len(T)}")
if len(FIN_DF) >= 3:
    L.append("")
    L.append("IL TAGLIO E' REALE?")
    L.append(f"  finestra x{FIN_DF['N'].iloc[-1]/FIN_DF['N'].iloc[0]:.0f}, "
             f"tau_c x{FIN_DF['tau_c'].iloc[-1]/FIN_DF['tau_c'].iloc[0]:.2f}")
L.append("")
L.append("AVVERTENZE")
L.append("  - mu e tau_c sono fortemente correlati: mu non va riportato da solo.")
L.append("  - gli IC sono di profilo, non asintotici, proprio per questo motivo.")
L.append("  - Hill assume una potenza PURA e qui e' tenuto solo come diagnostica:")
L.append("    l'assenza di plateau e' cio' che motiva il modello troncato.")
sintesi = "\n".join(L)
(OUT_DIR / "sintesi.txt").write_text(sintesi, encoding="utf-8")
print(sintesi)